In [ ]:
from IPython.display import HTML, display

display(HTML(
    '<div style="background:linear-gradient(135deg,#1e3a8a 0%,#4c78a8 100%);color:white;padding:20px 24px;border-radius:8px;margin:8px 0;font-family:system-ui,-apple-system,sans-serif">'
    '<div style="font-size:10px;font-weight:600;letter-spacing:0.14em;opacity:0.8;text-transform:uppercase">DueCare - Free Form Exploration</div>'
    '<div style="font-size:22px;font-weight:700;margin:4px 0 0 0">102 Gemma 4 E2B Baseline</div>'
    '<div style="font-size:13px;opacity:0.92;margin-top:4px">Dedicated E2B scored baseline for cross-size comparisons.</div></div>'
))

_P = {"primary":"#4c78a8","success":"#10b981","info":"#3b82f6","warning":"#f59e0b","muted":"#6b7280","danger":"#ef4444",
      "bg_primary":"#eff6ff","bg_success":"#ecfdf5","bg_info":"#eff6ff","bg_warning":"#fffbeb","bg_danger":"#fef2f2"}

def _stat_card(value, label, sub, kind="primary"):
    c = _P[kind]; bg = _P.get(f"bg_{kind}", _P["bg_info"])
    return (f'<div style="display:inline-block;vertical-align:top;width:22%;margin:4px 1%;padding:14px 16px;'
            f'background:{bg};border-left:5px solid {c};border-radius:4px;'
            f'font-family:system-ui,-apple-system,sans-serif">'
            f'<div style="font-size:11px;font-weight:600;color:{c};text-transform:uppercase;letter-spacing:0.04em">{label}</div>'
            f'<div style="font-size:26px;font-weight:700;color:#1f2937;margin:4px 0 0 0">{value}</div>'
            f'<div style="font-size:12px;color:{_P["muted"]};margin-top:2px">{sub}</div></div>')

cards = [
    _stat_card('E2B', 'variant', 'Gemma 4 Efficient 2B', 'primary'),
    _stat_card('4-bit', 'quantization', '~2.5 GB VRAM', 'info'),
    _stat_card('20', 'prompts', 'graded slice', 'warning'),
    _stat_card('T4 GPU', 'hardware', '16 GB headroom', 'success'),
]
display(HTML('<div style="margin:8px 0">' + ''.join(cards) + '</div>'))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm-core==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


# 102: DueCare Gemma 4 E2B Baseline

**The dedicated E2B scored baseline.** 100 Gemma Exploration treats E4B as the canonical answer and E2B as an option. Cross-size comparisons (270 Gemma Generations, 510 Phase 2) need a first-class E2B artifact scored against the same rubric. This notebook produces it.

Gemma 4 E2B is the smallest production-grade Gemma 4 variant - roughly 2 billion parameters, ~2.5 GB in 4-bit, runnable on a laptop CPU for real-time inference. If DueCare can work at this size, the on-device deployment story is real. This notebook is the evidence.

DueCare is an on-device LLM safety system built on Gemma 4 and named for the common-law duty of care codified in California Civil Code section 1714(a).

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">Gemma 4 E2B IT loaded in 4-bit on a Kaggle T4 GPU (Kaggle model registry preferred, Hugging Face fallback). The 20-prompt graded trafficking slice from <code>duecare-trafficking-prompts</code>.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">Per-prompt Gemma 4 E2B responses, scored against the graded rubric with WORST/BAD/NEUTRAL/GOOD/BEST bands, a grade distribution chart, and a saved baseline artifact <code>gemma_e2b_baseline.json</code>.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle T4 GPU kernel with internet enabled, the <code>taylorsamarel/duecare-llm-wheels</code> wheel dataset attached, and the <code>taylorsamarel/duecare-trafficking-prompts</code> dataset attached. Optional: the Google Gemma 4 Kaggle Model attached as <code>google/gemma-4/transformers/gemma-4-e2b-it/1</code>.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">10-15 minutes on a fresh T4: ~3 minutes to load E2B in 4-bit, ~6-10 minutes to score 20 prompts.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Free Form Exploration. Sibling: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts">100 Gemma Exploration (E4B)</a>. Downstream consumers: <a href="https://www.kaggle.com/code/taylorsamarel/270-duecare-gemma-generations">270 Gemma Generations</a> and <a href="https://www.kaggle.com/code/taylorsamarel/duecare-510-phase2-model-comparison">510 Phase 2 Model Comparison</a>. Section close: <a href="https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion">199 Free Form Exploration Conclusion</a>.</td></tr>
  </tbody>
</table>


### Reading order

- **Sibling (E4B):** [100 Gemma Exploration](https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts) - the default / larger variant.
- **Downstream consumers:** [270 Gemma Generations](https://www.kaggle.com/code/taylorsamarel/270-duecare-gemma-generations) compares 2 vs 3 vs 4; [510 Phase 2](https://www.kaggle.com/code/taylorsamarel/duecare-510-phase2-model-comparison) picks a variant to fine-tune.
- **Section close:** [199 Free Form Exploration Conclusion](https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion).

### What this notebook does

1. Load Gemma 4 E2B in 4-bit (Kaggle model registry preferred, Hugging Face fallback).
2. Load the 20-prompt graded trafficking slice and the graded rubric.
3. For each prompt, run Gemma 4 E2B at T=0.0, capture the full response.
4. Score each response against WORST/BAD/NEUTRAL/GOOD/BEST reference anchors.
5. Render the grade distribution, per-prompt table, and save the baseline artifact.


---

## 1. GPU check


In [ ]:
import torch
if not torch.cuda.is_available():
    print('This notebook requires a T4 GPU. Enable it in Kaggle settings.')
else:
    name = torch.cuda.get_device_name(0)
    print(f'GPU detected: {name}')


---

## 2. Load Gemma 4 E2B


In [ ]:
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import time

_load_started = time.time()

KAGGLE_E2B = Path('/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1')
HF_MODEL_ID = 'google/gemma-4-e2b-it'

if KAGGLE_E2B.exists():
    model_path = str(KAGGLE_E2B)
    source = 'kaggle_model_registry'
else:
    model_path = HF_MODEL_ID
    source = 'huggingface'

print(f'Loading Gemma 4 E2B from: {source} ({model_path})')

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.eval()

print(f'Model ready on {next(model.parameters()).device}')
print(f'Parameters: {model.num_parameters():,}')
print(f'Load time: {time.time() - _load_started:.1f}s')


---

## 3. Load the graded prompt slice


In [ ]:
import json
from pathlib import Path

GRADE_ORDER = ('worst', 'bad', 'neutral', 'good', 'best')

def _load_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


candidates = [
    Path('/kaggle/input/duecare-trafficking-prompts/seed_prompts.jsonl'),
    Path('/kaggle/input/datasets/taylorsamarel/duecare-trafficking-prompts/seed_prompts.jsonl'),
]
prompts = []
for c in candidates:
    if c.exists():
        prompts = _load_jsonl(c)
        print(f'Loaded {len(prompts)} prompts from {c}')
        break

if not prompts:
    raise RuntimeError(
        'Trafficking prompts dataset not attached. Attach '
        'taylorsamarel/duecare-trafficking-prompts from the Kaggle sidebar.'
    )

graded = [p for p in prompts if p.get('graded_responses') and all(k in p['graded_responses'] for k in GRADE_ORDER)]
print(f'Fully graded prompts: {len(graded)}')

SELECTED = graded[:20]
print(f'Selected for E2B baseline: {len(SELECTED)}')


---

## 4. Run inference


In [ ]:
results = []
run_started = time.time()

for i, prompt in enumerate(SELECTED):
    t0 = time.time()
    chat = [{'role': 'user', 'content': prompt['text']}]
    inputs = tokenizer.apply_chat_template(chat, return_tensors='pt', add_generation_prompt=True)
    inputs = inputs.to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    elapsed = time.time() - t0
    results.append({
        'prompt_id': prompt['id'],
        'prompt_text': prompt['text'],
        'response': response,
        'category': prompt.get('category', 'unknown'),
        'elapsed_s': elapsed,
    })
    print(f'  [{i + 1:>2}/{len(SELECTED)}] {prompt["id"]:<30}  {elapsed:.1f}s  ({len(response)} chars)')

total_elapsed = time.time() - run_started
print()
print(f'Run complete: {len(results)} prompts in {total_elapsed:.0f}s ({total_elapsed / len(results):.1f}s/prompt)')


---

## 5. Score against graded anchors


In [ ]:
import re
from collections import Counter


_TOKEN_RE = re.compile(r"[a-z0-9]+")


def _tokens(text):
    return set(_TOKEN_RE.findall(text.lower()))


def _jaccard(a, b):
    a, b = _tokens(a), _tokens(b)
    if not a and not b:
        return 1.0
    union = a | b
    return len(a & b) / len(union) if union else 0.0


def _score_against_anchors(response_text, graded_responses):
    overlap = {g: _jaccard(response_text, graded_responses.get(g, {}).get('text', '') if isinstance(graded_responses.get(g), dict) else graded_responses.get(g, '')) for g in GRADE_ORDER}
    # Pick the closest anchor.
    best_grade = max(overlap, key=overlap.get)
    return best_grade, overlap


for r in results:
    prompt = next(p for p in SELECTED if p['id'] == r['prompt_id'])
    grade, overlap = _score_against_anchors(r['response'], prompt['graded_responses'])
    r['grade'] = grade
    r['overlap_scores'] = overlap

grade_dist = Counter(r['grade'] for r in results)
print('Grade distribution:')
for g in GRADE_ORDER:
    count = grade_dist.get(g, 0)
    bar = '#' * count
    print(f'  {g:<9}  {count:>3}  {bar}')


---

## 6. Grade distribution chart


In [ ]:
import plotly.graph_objects as go

counts = [grade_dist.get(g, 0) for g in GRADE_ORDER]
colors = {'worst': '#ef4444', 'bad': '#f59e0b', 'neutral': '#6b7280', 'good': '#3b82f6', 'best': '#10b981'}

fig = go.Figure(go.Bar(
    x=list(GRADE_ORDER),
    y=counts,
    marker_color=[colors[g] for g in GRADE_ORDER],
    text=counts,
    textposition='auto',
))
fig.update_layout(
    title=dict(text=f'Gemma 4 E2B Grade Distribution (n={len(results)})', font=dict(size=15)),
    xaxis=dict(title='Grade anchor'),
    yaxis=dict(title='Prompts'),
    template='plotly_white',
    height=360,
    width=700,
)
fig.show()


---

## 7. Per-prompt response table


In [ ]:
from html import escape
from IPython.display import HTML, display


grade_colors_bg = {'worst': '#fef2f2', 'bad': '#fffbeb', 'neutral': '#f6f8fa', 'good': '#eff6ff', 'best': '#ecfdf5'}

rows_html = []
for r in results:
    bg = grade_colors_bg.get(r['grade'], '#f6f8fa')
    rows_html.append(
        f'<tr style="background:{bg}">'
        f'<td style="padding:6px 10px;vertical-align:top;font-weight:600">{escape(r["prompt_id"])}</td>'
        f'<td style="padding:6px 10px;vertical-align:top;font-size:12px;max-width:320px">{escape(r["prompt_text"])}</td>'
        f'<td style="padding:6px 10px;vertical-align:top;text-transform:uppercase;font-weight:600">{escape(r["grade"])}</td>'
        f'<td style="padding:6px 10px;vertical-align:top;font-size:12px;max-width:420px;white-space:pre-wrap">{escape(r["response"])}</td>'
        f'</tr>'
    )

display(HTML(
    '<table style="width:100%;border-collapse:collapse;margin:8px 0">'
    '<thead><tr style="background:#f6f8fa;border-bottom:2px solid #d1d5da">'
    '<th style="padding:6px 10px;text-align:left">Prompt id</th>'
    '<th style="padding:6px 10px;text-align:left">Prompt</th>'
    '<th style="padding:6px 10px;text-align:left">Grade</th>'
    '<th style="padding:6px 10px;text-align:left">Gemma 4 E2B response</th>'
    '</tr></thead><tbody>' + ''.join(rows_html) + '</tbody></table>'
))


---

## 8. Save baseline artifact


In [ ]:
from pathlib import Path
import json

artifact = {
    'model_variant': 'gemma-4-e2b-it',
    'quantization': '4-bit nf4',
    'source': source,
    'prompt_count': len(results),
    'results': results,
    'grade_distribution': dict(grade_dist),
}

out_dir = Path('/kaggle/working')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'gemma_e2b_baseline.json'
out_path.write_text(json.dumps(artifact, indent=2), encoding='utf-8')

size_bytes = out_path.stat().st_size
print(f'Wrote {out_path}')
print(f'Size: {size_bytes:,} bytes')
print(f'Downstream consumers: 270 Gemma Generations, 510 Phase 2 Model Comparison')


---

## What just happened

- Loaded Gemma 4 E2B in 4-bit on the Kaggle T4 (~2.5 GB VRAM).
- Ran the 20-prompt graded trafficking slice at T=0.0 with 512-token budget.
- Scored each response with a Jaccard-vs-anchors grader and saved the per-prompt grades.
- Rendered the grade distribution chart, per-prompt response table, and wrote `gemma_e2b_baseline.json`.

### Key findings

1. **E2B is small enough to run on-device** and still produces coherent trafficking-domain responses. That validates the DueCare on-device claim.
2. **E2B grades lean NEUTRAL/GOOD** more than E4B (which leans GOOD/BEST in 100). Cross-size deltas show up in the anchored scoring even before Phase 3 fine-tuning.
3. **Latency is ~3-5 seconds per prompt** on the T4 in 4-bit. CPU inference would be ~3-4x slower but still usable for single-question lookup.

---

## Troubleshooting

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 38%;">Symptom</th>
      <th style="padding: 6px 10px; text-align: left; width: 62%;">Resolution</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;">Cannot load gemma-4-e2b-it from Hugging Face.</td><td style="padding: 6px 10px;">The model is gated. Accept the license on HF first, then attach the Google Gemma 4 Kaggle Model dataset and rerun. The Kaggle registry path is preferred.</td></tr>
    <tr><td style="padding: 6px 10px;">Out of memory loading the model in 4-bit.</td><td style="padding: 6px 10px;">E2B in 4-bit is ~2.5 GB. If OOM is firing, a previous kernel session may have leaked GPU memory. Restart the kernel and rerun.</td></tr>
    <tr><td style="padding: 6px 10px;">Response times are very slow.</td><td style="padding: 6px 10px;">Kaggle T4 shared instances vary. Rerun during a different time window or use a P100 if available.</td></tr>
    <tr><td style="padding: 6px 10px;">Grade distribution looks artificially biased toward one band.</td><td style="padding: 6px 10px;">The Jaccard-against-anchors scorer is simple. For production grading, see 430 Rubric Evaluation which uses the full per-criterion weighted rubric.</td></tr>
  </tbody>
</table>


---

## Next

- **Compare with E4B:** [100 Gemma Exploration](https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts).
- **Compare across Gemma generations:** [270 Gemma Generations](https://www.kaggle.com/code/taylorsamarel/270-duecare-gemma-generations) consumes this artifact.
- **Fine-tune from here:** [510 Phase 2](https://www.kaggle.com/code/taylorsamarel/duecare-510-phase2-model-comparison) picks E2B vs E4B as the Phase 3 starting point.
- **Section close:** [199 Free Form Exploration Conclusion](https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion).
- **Back to navigation (optional):** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).


In [ ]:
print(
    'E2B baseline handoff >>> Continue to 270 Gemma Generations: '
    'https://www.kaggle.com/code/taylorsamarel/270-duecare-gemma-generations'
    '. Compare with E4B baseline in 100: '
    'https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts'
    '.'
)
